In [ ]:
from pathlib import Path
import sys

project_dir = Path.cwd() if (Path.cwd() / 'dataset.py').exists() else Path.cwd().parent
sys.path.insert(0, str(project_dir))

from modules import Embedder, Decoder
from dataset import LOBSTERLevel10Dataset
from torch.nn.modules.loss import MSELoss
from torch.utils.data import DataLoader
from torch import optim
from itertools import chain
import torch

EPOCHS = 15

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

embedder = Embedder().to(device)
decoder = Decoder.to(device)

train_data = LOBSTERLevel10Dataset(train=True, normalize=True)
validation_data = LOBSTERLevel10Dataset(train=False, normalize=True)
train_loader = DataLoader(train_data, shuffle=True, batch_size=1024)

optimizer = optim.Adam(chain(embedder.parameters(), decoder.parameters()), lr=3e-4)
loss = MSELoss()

In [ ]:
embedder.train()
decoder.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for datum in train_loader:
        datum = datum.to(device)
        optimizer.zero_grad()
        z = embedder(datum)
        recon = decoder(z)
        recon_loss = loss(datum, recon)
        recon_loss.backward()
        optimizer.step()
        total_loss += recon_loss.item()
    print(f'Epoch {epoch+1} done, total reconstruction loss = {total_loss:1.7f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

embedder.eval()
decoder.eval()
results = {}
with torch.no_grad():
    for split, data in (('Training', train_data), ('Validation', validation_data)):
        features = torch.cat([
            decoder(embedder(batch.to(device))).cpu()
            for batch in DataLoader(data, shuffle=False, batch_size=1024)
        ])
        results[split] = (data.books.reset_index(drop=True),
                          data.feature_sequence_to_df(features))

# Metrics use each complete split; plots use short windows for legibility.
fig, axes = plt.subplots(
    3, 2, figsize=(17, 10), sharex='col', layout='constrained',
    gridspec_kw={'height_ratios': [4, 1, 1]},
)
plot_length = 400
for column, (split, (books, reconstructed_books)) in enumerate(results.items()):
    midpoints = (books['ASKp1'] + books['BIDp1']) / 2
    midpoints_recon = (reconstructed_books['ASKp1'] + reconstructed_books['BIDp1']) / 2
    spreads = books['ASKp1'] - books['BIDp1']
    spreads_recon = reconstructed_books['ASKp1'] - reconstructed_books['BIDp1']
    size_columns = [name for name in books.columns if 's' in name]
    print(f'{split} midpoint MAE: ${np.abs(midpoints_recon - midpoints).mean() / 10_000:.5f}')
    print(f'{split} spread MAE: ${np.abs(spreads_recon - spreads).mean() / 10_000:.5f}')
    print(f'{split} size MAE: {(books[size_columns] - reconstructed_books[size_columns]).abs().to_numpy().mean():.2f} shares')

    plot_start = 20_000 if split == 'Training' else 0
    window = slice(plot_start, plot_start + plot_length)
    x = np.arange(len(books))[window]
    ax_book, ax_mid, ax_spread = axes[:, column]
    for book, label, color in ((books, 'Original', 'tab:blue'),
                               (reconstructed_books, 'Reconstructed', 'tab:orange')):
        for level, alpha in ((10, 0.08), (5, 0.16), (1, 0.28)):
            ax_book.fill_between(
                x, book[f'BIDp{level}'].iloc[window] / 10_000,
                book[f'ASKp{level}'].iloc[window] / 10_000,
                color=color, alpha=alpha, label=label if level == 1 else None,
            )
        for side in ('BID', 'ASK'):
            ax_book.plot(x, book[f'{side}p1'].iloc[window] / 10_000,
                         color=color, linewidth=0.9)
    ax_book.set(title=f'{split}: levels 10, 5, 1 (increasing opacity)', ylabel='Price ($)')
    ax_book.legend()
    ax_mid.plot(x, (midpoints_recon - midpoints).iloc[window] / 10_000, color='tab:red')
    ax_mid.axhline(0, color='black', linewidth=0.7)
    ax_mid.set(ylabel='Midpoint error ($)')
    ax_spread.plot(x, (spreads_recon - spreads).iloc[window] / 10_000, color='tab:purple')
    ax_spread.axhline(0, color='black', linewidth=0.7)
    ax_spread.set(xlabel='Event index within split', ylabel='Spread error ($)')
plt.show()
results['Validation'][1].head()